In [1]:
# Author: Chris Benjamin
# S&P 500 Direction Classifier with VIX/Fear, Ratio & Trend Features
# Data: ^GSPC and ^VIX from Yahoo Finance via yfinance
# Target: 1 if next day's close > today's close, else 0
# Method: Walk-forward backtest with RandomForest (tuned) and optional Logistic baseline
# Notes: All timestamps coerced to timezone-naive; feature windows use trading days
# Last updated: <10/10/25>


In [2]:
import yfinance as yf
import pandas as pd

In [3]:
!pip3 install numpy

In [4]:
import sys

In [5]:
# Download full S&P 500 (^GSPC) history as a ticker object
sp500 = yf.Ticker("^GSPC")

In [6]:
# Conver ticker -> price history dataframe (daily OHLCV)
sp500 = sp500.history(period="max")

In [7]:
del sp500["Dividends"]
del sp500["Stock Splits"]

In [8]:
# Label engieering: tommorow's close aligned to today's row. For training.
sp500["Tomorrow"] = sp500["Close"].shift(-1)

In [9]:
# Classification target: 1 is tomorrow's close > today's close; 0 otherwise
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [10]:
#removes all rows before Jan 1, 1985
sp500 = sp500.loc["1985-01-01":].copy()

In [11]:
# Ensure timezone-naive index to avoid merge or align issues later
sp500.index = sp500.index.tz_localize(None)

In [12]:
sp500 = sp500.copy()

In [13]:
# VIX download & alignment
# 1) Guarantee DatetimeIndex on S&P data
# 2) Download ^VIX over the same calendar span
sp500 = sp500.copy()
if not isinstance(sp500.index, pd.DatetimeIndex):
    sp500 = sp500.set_index(pd.to_datetime(sp500["Date"]))

import yfinance as yf
vix_raw = yf.download("^VIX",
                      start=sp500.index.min().date(),
                      end=(sp500.index.max() + pd.Timedelta(days=1)).date(),
                      progress=False, auto_adjust=False)


In [14]:
# Keep only VIX close, make index tz-naive, and align to S&P 500 trading days
vix_series = vix_raw[["Close"]].squeeze().rename("VIX").copy()
vix_series.index = pd.to_datetime(vix_series.index).tz_localize(None)

sp500.index = pd.to_datetime(sp500.index).tz_localize(None)
sp500["VIX"] = vix_series.reindex(sp500.index)

In [15]:
# 3) Create VIX features
sp500 = sp500.copy()  # silence SettingWithCopy warnings if sp500 was sliced earlier
sp500.loc[:, "VIX_Lag1"]    = sp500["VIX"].shift(1)
sp500.loc[:, "VIX_Change"]  = sp500["VIX"].pct_change(fill_method=None)
sp500.loc[:, "VIX_5d"]      = sp500["VIX"].rolling(5, min_periods=1).mean()
sp500.loc[:, "VIX_21d_vol"] = sp500["VIX"].pct_change(fill_method=None).rolling(21).std()

In [16]:
# "Fear sentiment" feature set derived from VIX:
# - Fear_Zscore: standardized deviation from trailing 1y mean
# - Fear_High: VIX > 20 indicator (crude high-volatility regime)
# - Fear_STLT_diff: 5d VIX mean minus 1y mean (short vs long-term tension)

sp500 = sp500.copy()
sp500["Fear_Zscore"] = (
    (sp500["VIX"] - sp500["VIX"].rolling(252, min_periods=20).mean()) /
    sp500["VIX"].rolling(252, min_periods=20).std()
)

# 2) Binary indicator — market in "fear" mode (VIX > 20)
sp500["Fear_High"] = (sp500["VIX"] > 20).astype(int)

# 3) Short-term vs long-term volatility difference
sp500["Fear_STLT_diff"] = (
    sp500["VIX"].rolling(5, min_periods=1).mean() -
    sp500["VIX"].rolling(252, min_periods=20).mean()
)

In [17]:
# Multi-horizon price structure features
# - Close_Ratio_{horizon}: Close / {horizon}-day rolling mean (mean reversion vs momentum)
# - Trend_{horizon}: count of positive days in prior {horizon} days (shifted to avoid leakage)

horizons = [2,5,60,250,1000]
fear_features = ["Fear_Zscore", "Fear_High", "Fear_STLT_diff"]
predictors3 = []
predictors3.extend(fear_features)

for horizon in horizons:
    rolling_average = sp500.rolling(horizon).mean()  # rolling h-day means

    ratio_column = f"Close_Ratio_{horizon}"
    sp500.loc[:, ratio_column] = sp500["Close"] / rolling_average["Close"]  # ratio between close col and rolling means

    trend_column = f"Trend_{horizon}"
    sp500.loc[:, trend_column] = sp500["Target"].shift(1).rolling(horizon).sum()  # count of prior day green (sp500 went up) count

    predictors3 += [ratio_column, trend_column]  # add to predictors for modeling


In [18]:
sp500 = sp500.dropna()

In [19]:
import sklearn

In [20]:
# === Imports ===
import numpy as np

# ML tools for time-series modeling and evaluation
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# === Feature setup ===
# Base predictors: raw OHLCV market features
predictors = ["Close", "Volume", "Open", "High", "Low"]

# Add engineered features such as ratios, trends, and fear index metrics
predictors += predictors3

# Drop any rows with missing values in predictors or target variable
df = sp500.dropna(subset=predictors + ["Target"]).copy()

# Split dataframe into feature matrix X and target vector y
X = df[predictors].values
y = df["Target"].values

# === Train–test split ===
# Reserve the most recent 250 trading days as an unseen holdout set
HOLDOUT = 250
X_train, y_train = X[:-HOLDOUT], y[:-HOLDOUT]
X_test,  y_test  = X[-HOLDOUT:], y[-HOLDOUT:]

# === Time-aware cross-validation ===
# Ensures train/test splits preserve chronological order (no lookahead bias)
tscv = TimeSeriesSplit(n_splits=5)

# === Random Forest tuning ===
# Initialize Random Forest and perform grid search to find best hyperparameters
rf = RandomForestClassifier(random_state=1, n_jobs=-1)
param_grid = {
    "n_estimators": [120, 180, 300],        # number of trees in the forest
    "min_samples_split": [25, 50, 75, 100], # min samples to split internal nodes
    "max_depth": [None, 8, 12],             # tree depth (None = unlimited)
}

# Grid search across parameter combinations using precision as the scoring metric
gcv = GridSearchCV(
    rf,
    param_grid,
    cv=tscv,
    scoring="precision",
    n_jobs=-1,
    refit=True
)
gcv.fit(X_train, y_train)
best_rf = gcv.best_estimator_
print("Best RF params:", gcv.best_params_)

# === Logistic Regression baseline ===
# Baseline linear model with scaling and logistic regression
logit = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, solver="lbfgs"))
])
logit.fit(X_train, y_train)

# === Threshold sweep function ===
# Evaluates model performance over a range of probability thresholds
# Returns a DataFrame sorted by precision for easy comparison
def threshold_sweep(model, X_te, y_te, thresholds=np.arange(0.50, 0.71, 0.02)):
    p = model.predict_proba(X_te)[:, 1]  # predicted probability of "up" (class 1)
    rows = []
    for t in thresholds:
        yhat = (p >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "precision": float(precision_score(y_te, yhat, zero_division=0)),
            "recall": float(recall_score(y_te, yhat, zero_division=0)),
            "f1": float(f1_score(y_te, yhat, zero_division=0)),
        })
    return pd.DataFrame(rows).sort_values("precision", ascending=False)

# Run threshold sweeps for both models on the holdout set
rf_sweep = threshold_sweep(best_rf, X_test, y_test)
lg_sweep = threshold_sweep(logit,   X_test, y_test)

print("\nRandomForest holdout sweep (sorted by precision):")
display(rf_sweep.head(10))

print("\nLogistic baseline holdout sweep (sorted by precision):")
display(lg_sweep.head(10))

# === Final evaluation at chosen threshold ===
THRESH = 0.60  # decision threshold for classifying an "up" day

# Function to compute precision, recall, F1, and confusion matrix at given threshold
def eval_at_threshold(model, X_te, y_te, t=THRESH):
    p = model.predict_proba(X_te)[:, 1]
    yhat = (p >= t).astype(int)
    return {
        "threshold": t,
        "precision": precision_score(y_te, yhat, zero_division=0),
        "recall": recall_score(y_te, yhat, zero_division=0),
        "f1": f1_score(y_te, yhat, zero_division=0),
        "confusion_matrix": confusion_matrix(y_te, yhat).tolist(),
    }

# Print summary performance for both models at threshold 0.60
print("\nRF @ 0.60:", eval_at_threshold(best_rf, X_test, y_test, THRESH))
print("Logit @ 0.60:", eval_at_threshold(logit,   X_test, y_test, THRESH))


Best RF params: {'max_depth': 8, 'min_samples_split': 100, 'n_estimators': 120}

RandomForest holdout sweep (sorted by precision):


,threshold,precision,recall,f1
2,0.54,0.580000,0.204225,0.302083
0,0.50,0.572165,0.781690,0.660714
1,0.52,0.555556,0.563380,0.559441
3,0.56,0.375000,0.021127,0.040000
4,0.58,0.000000,0.000000,0.000000
5,0.60,0.000000,0.000000,0.000000
6,0.62,0.000000,0.000000,0.000000
7,0.64,0.000000,0.000000,0.000000
8,0.66,0.000000,0.000000,0.000000
9,0.68,0.000000,0.000000,0.000000



Logistic baseline holdout sweep (sorted by precision):


,threshold,precision,recall,f1
5,0.60,1.000000,0.042254,0.081081
6,0.62,1.000000,0.007042,0.013986
4,0.58,0.620690,0.126761,0.210526
3,0.56,0.597403,0.323944,0.420091
0,0.50,0.571429,0.985915,0.723514
1,0.52,0.566038,0.845070,0.677966
2,0.54,0.551020,0.570423,0.560554
7,0.64,0.000000,0.000000,0.000000
8,0.66,0.000000,0.000000,0.000000
9,0.68,0.000000,0.000000,0.000000



RF @ 0.60: {'threshold': 0.6, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'confusion_matrix': [[108, 0], [142, 0]]}
Logit @ 0.60: {'threshold': 0.6, 'precision': 1.0, 'recall': 0.04225352112676056, 'f1': 0.08108108108108109, 'confusion_matrix': [[108, 0], [136, 6]]}


In [21]:
sp500 = sp500.dropna()

In [22]:
def predict(train, test, predictors, model, threshold=0.60):
    """
    Fits a given classification model on a training dataset and generates 
    probability-based predictions on a test dataset.
    
    Parameters:
        train (DataFrame): Training data 
        test (DataFrame):  Test data 
        predictors (list): List of feature column names used for training.
        model (object):    Any fitted sklearn-style model supporting predict_proba().
        threshold (float): Decision cutoff for classifying an "up" day (default=0.60).

    Returns:
        DataFrame: A copy of the test set containing:
            - "Target": the true class label (0 or 1)
            - "Proba":  model-predicted probability of class 1
            - "Predictions": binary prediction based on threshold
    """
    

    model.fit(train[predictors], train["Target"])
    
    # Predict the probability of the positive class (class 1) for the test set
    proba = model.predict_proba(test[predictors])[:, 1]
    out = test[["Target"]].copy()
    
    # Store the model's predicted probabilities
    out["Proba"] = proba
    
    # Convert probabilities to binary predictions using the specified threshold
    out["Predictions"] = (proba >= threshold).astype(int)
    

    return out


In [23]:
def backtest(data, model, predictors, start=3000, step=250):
    """
    Performs a walk-forward (rolling) backtest to evaluate a predictive model 
    on time-series data without data leakage.

    Parameters:
        data (DataFrame):   Full dataset containing predictors and "Target".
        model (object):     Any sklearn-style model supporting fit() and predict_proba().
        predictors (list):  List of feature column names used for training.
        start (int):        Initial index to begin testing (default=3000). 
                            The model is first trained on data up to this index (day).
        step (int):         Number of rows (days) to move forward in each backtest iteration.
                            Each iteration trains on all prior data and tests on the next 'step' window.

    Returns:
        DataFrame: Combined predictions for all test periods, including:
            - "Target": true next-day direction (0 or 1)
            - "Proba":  predicted probability of upward movement
            - "Predictions": model's binary classification
    """
    
    all_predictions = []

    # Iterate through the dataset in increments of 'step'
    # Each loop trains the model on all data up to 'i' and tests on the following 'step' days
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        
        # Generate predictions using the previously defined 'predict()' function
        predictions = predict(train, test, predictors, model)
        
        # Store the resulting predictions in all_predictions
        all_predictions.append(predictions)
    
    # Concatenate all individual test-period results into one DataFrame
    return pd.concat(all_predictions)


In [24]:

TUNED_RF = best_rf     
THRESH   = 0.60        # threshold for percent certainty that the S&P will go up/down

def backtest_with_threshold(data, model, predictors, threshold=0.60, start=3000, step=250):
    """
    Performs a walk-forward (rolling) backtest that repeatedly trains a model 
    on historical data and evaluates it on future segments, using a specified 
    probability threshold to classify predictions.

    Parameters:
        data (DataFrame):   Full dataset containing predictors and "Target".
        model (object):     Any sklearn-style model supporting fit() and predict_proba().
        predictors (list):  List of feature column names used for training.
        threshold (float):  Probability cutoff for predicting an "up" day (default=0.60).
        start (int):        Initial index to begin testing (default=3000). 
                            The model is first trained on data up to this index (day).
        step (int):         Number of rows (days) to move forward in each backtest iteration.
                            Each iteration trains on all prior data and tests on the next 'step' window.

    

    Returns:
        DataFrame: Combined predictions for all test segments, including:
            - "Target": actual class (0 or 1)
            - "Proba":  predicted probability of an upward move
            - "Predictions": binary model output based on the threshold
    """

    # List to hold predictions from each backtest segment
    all_predictions = []

    # Walk-forward loop — retrain and test model sequentially through time
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()

        # Generate model predictions using custom threshold for classification
        predictions = predict(train, test, predictors, model, threshold=threshold)

        # Store each segment’s predictions
        all_predictions.append(predictions)

    # If no predictions were generated (e.g., not enough rows), return empty DataFrame
    if len(all_predictions) == 0:
        return pd.DataFrame()

    # Combine predictions from all time segments into one DataFrame
    return pd.concat(all_predictions)


In [25]:
# Run Walk-forward backtest with tunned RF at 0.60 threshold.
bt_preds = backtest_with_threshold(sp500, TUNED_RF, predictors, start=3000, step=250, threshold=THRESH)

from sklearn.metrics import precision_score

# Primary performance metric indicating the model’s effectiveness/accuracy on the out-of-sample backtest
precision_score(bt_preds["Target"], bt_preds["Predictions"])


0.5467980295566502

In [26]:
predictors

['Close',
 'Volume',
 'Open',
 'High',
 'Low',
 'Fear_Zscore',
 'Fear_High',
 'Fear_STLT_diff',
 'Close_Ratio_2',
 'Trend_2',
 'Close_Ratio_5',
 'Trend_5',
 'Close_Ratio_60',
 'Trend_60',
 'Close_Ratio_250',
 'Trend_250',
 'Close_Ratio_1000',
 'Trend_1000']

In [27]:
sp500

,Open,High,Low,Close,Volume,Tomorrow,Target,VIX,VIX_Lag1,VIX_Change,...,Close_Ratio_2,Trend_2,Close_Ratio_5,Trend_5,Close_Ratio_60,Trend_60,Close_Ratio_250,Trend_250,Close_Ratio_1000,Trend_1000
Date,,,,,,,,,,,,,,,,,,,,,
1990-01-31,322.980011,329.079987,322.980011,329.079987,189660000,328.790009,0,25.360001,27.250000,-0.069358,...,1.009355,1.0,1.009981,1.0,0.959505,32.0,1.003247,141.0,1.168535,560.0
1990-02-01,329.079987,329.859985,327.760010,328.790009,154580000,330.920013,1,24.870001,25.360001,-0.019322,...,0.999559,1.0,1.007415,1.0,0.958837,32.0,1.001963,141.0,1.167054,560.0
1990-02-02,328.790009,332.100006,328.089996,330.920013,164400000,331.850006,1,24.320000,24.870001,-0.022115,...,1.003229,1.0,1.010770,2.0,0.965231,32.0,1.008069,141.0,1.174161,560.0
1990-02-05,330.920013,332.160004,330.450012,331.850006,130950000,329.660004,0,24.540001,24.320000,0.009046,...,1.001403,2.0,1.009510,3.0,0.968240,32.0,1.010493,142.0,1.177013,560.0
1990-02-06,331.850006,331.859985,328.200012,329.660004,134070000,333.750000,1,24.690001,24.540001,0.006112,...,0.996689,1.0,0.998788,3.0,0.962174,32.0,1.003414,142.0,1.168809,560.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-03,6722.140137,6750.870117,6705.669922,6715.790039,5713110000,6740.279785,1,16.650000,16.629999,0.001203,...,1.000033,2.0,1.002596,5.0,1.039023,35.0,1.118160,143.0,1.377161,532.0
2025-10-06,6733.859863,6749.520020,6717.779785,6740.279785,5604460000,6714.589844,0,16.370001,16.650000,-0.016817,...,1.001820,2.0,1.003882,5.0,1.041522,36.0,1.121499,143.0,1.381509,533.0
2025-10-07,6746.140137,6754.490234,6699.959961,6714.589844,5546150000,6753.720215,1,17.240000,16.370001,0.053146,...,0.998091,1.0,0.999278,4.0,1.036362,35.0,1.116467,143.0,1.375577,533.0
